# Workshop 3B: Digital Certificates & PKI Lab (Solutions Guide)

**Topic**: Concurrency, Operating Systems, & Security — Digital Certificates & Public Key Infrastructure (PKI)  
**Target Duration**: ~20 minutes  
**Objective**: Programmatically retrieve, inspect, and verify SSL/TLS certificates using Python's built-in `socket` and `ssl` libraries, understanding host verification and certificate chain validation.

---  
## Exercise 1: Programmatic SSL/TLS Certificate Retrieval

To establish a secure HTTPS connection, the client and server perform a TLS handshake. In this step, the client downloads the server's public digital certificate and validates it against its Root Store.

We will connect to `london.port.ac.uk` on port `443` to retrieve and inspect its certificate.

In [ ]:
import socket
import ssl

hostname = "london.port.ac.uk"
port = 443

# 1. Create a default SSL context (loads default Root certificates from the operating system)
context = ssl.create_default_context()

# 2. Establish connection and wrap the socket in SSL
with socket.create_connection((hostname, port)) as sock:
    with context.wrap_socket(sock, server_hostname=hostname) as ssock:
        # Retrieve the parsed certificate dictionary from the SSL socket
        cert = ssock.getpeercert()

# 3. Parse and Print metadata
if cert:
    # Convert subject and issuer tuple format into a readable dictionary
    subject_dict = dict(x[0] for x in cert.get('subject', []))
    issuer_dict = dict(x[0] for x in cert.get('issuer', []))
    
    print(f"--- Certificate Details for {hostname} ---")
    print(f"Common Name (Domain): {subject_dict.get('commonName')}")
    print(f"Issuer CA:            {issuer_dict.get('commonName')}")
    print(f"Valid From:           {cert.get('notBefore')}")
    print(f"Valid Until:          {cert.get('notAfter')}")
else:
    print("Could not retrieve certificate.")

---  
## Exercise 2: Extracting Subject Alternative Names (SANs) and Serial Number

A single SSL certificate can protect multiple subdomains or domains using the **Subject Alternative Name (SAN)** field. In this task, you will programmatically parse and list all alternative domains covered by the certificate.

In [ ]:
# Extract the Serial Number and Subject Alternative Names (SANs) from the 'cert' dictionary
serial_number = cert.get('serialNumber')
sans = cert.get('subjectAltName', [])

print(f"Certificate Serial Number: {serial_number}")
print("\nSubject Alternative Names (SANs):")
for san_type, san_value in sans:
    print(f"  - {san_type}: {san_value}")

---  
## Exercise 3: Observing SSL/TLS Verification Failures (Self-Signed Certificates)

If a server provides a certificate that is not signed by a CA in our trusted Root Store, or if the domain name doesn't match the certificate, the browser/client will reject the connection.

Let's test this by attempting to connect to `self-signed.badssl.com`.

In [ ]:
bad_hostname = "self-signed.badssl.com"

# Write a try-except block to connect to bad_hostname on port 443
context = ssl.create_default_context()

try:
    with socket.create_connection((bad_hostname, port)) as sock:
        with context.wrap_socket(sock, server_hostname=bad_hostname) as ssock:
            cert_bad = ssock.getpeercert()
except ssl.SSLCertVerificationError as e:
    print(f"Verification Failed (Expected!): {e}")
    print("Reason: The certificate presented is self-signed and not issued by a trusted root CA.")
except Exception as e:
    print(f"Other Connection Error: {e}")

---  
## Exercise 4: Automated Expiration Check Alerting Script

SSL certificates must be renewed frequently. Modern system administrators write monitoring scripts that parse certificate expiration dates and alert if a certificate is close to expiring.

Write a function `check_days_until_expiration(cert_dict)` that computes the remaining days before a certificate becomes invalid.

In [ ]:
from datetime import datetime

def check_days_until_expiration(cert_dict):
    not_after_str = cert_dict.get('notAfter')
    if not not_after_str:
        return None
    
    # Convert the 'notAfter' string date to a python datetime object
    expiry_date = datetime.strptime(not_after_str, "%b %d %H:%M:%S %Y %Z")
    days_remaining = (expiry_date - datetime.utcnow()).days
    
    return days_remaining

# Test using the certificate retrieved in Exercise 1
days_left = check_days_until_expiration(cert)
print(f"Days remaining until certificate expiration: {days_left} days")
if days_left < 30:
    print("WARNING: Certificate expires in less than 30 days! Action required.")
else:
    print("Certificate status: Active and secure.")

### Discussion Questions & Solutions
1. **Why is host name verification (matching the URL host to the certificate Common Name/SAN) a critical part of the TLS handshake?**
   * **Answer**: Because without host verification, an attacker could present a valid certificate issued for a domain they control (e.g. `malicious.com`) to intercept connections meant for a different host (e.g. `bank.com`). Checking that the domain requested matches the Common Name or SAN in the certificate guarantees that we are talking to the correct server and prevents Man-in-the-Middle (MitM) attacks.
2. **Why do certificates have short expiration dates (typically 397 days maximum today)?**
   * **Answer**: Shorter lifetimes limit the window of vulnerability if a certificate's private key is silently compromised. It also encourages rapid rotation of keys and ensures that inactive or revoked domains do not have active certificates indefinitely. Finally, it forces organizations to automate certificate renewal, reducing manual human error.